## Homework 12

In this homework, you are going to use the code from ```TrafficSignsClassification``` notebook and create your own traffic sign classifier.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

import pickle
from time import time
import numpy as np
from sklearn.utils import shuffle

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 6]

import keras
from keras import Model
from keras.layers import Dense, Input

### Step 1
Use the data from ```data/subset_homework.pckl``` (or the ```data/subset_homework``` folder) and visualize some examples. How many images are there for each class?

In [ ]:
with open('../data/subset_homework.pckl', 'rb') as f:
    data = pickle.load(f)

images_0 = data['class_id_0']
images_1 = data['class_id_1']
labels_0 = [0] * len(images_0)
labels_1 = [1] * len(images_1)

print('Image shape', images_0[0].shape, images_0[0].dtype)

In [ ]:
for row, (images, labels) in enumerate([(images_0, labels_0), (images_1, labels_1)]):
    for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
        plt.subplot(2, 10, row*10 + cnt + 1)
        plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
        plt.title(labels[idx]), plt.axis(False)

In [ ]:
print('Num samples class_0', len(images_0))
print('Num samples class_1', len(images_1))
print('Accuracy of always predicting class 0:', len(images_0) / (len(images_0) + len(images_1)))

In [ ]:
images = images_0 + images_1
labels = labels_0 + labels_1
images, labels = shuffle(images, labels)

images = np.array(images)
labels = np.array(labels)

print('Images', images.shape)
print('Labels', labels.shape)

In [ ]:
start = time()
pixels = np.array([image.flatten() for image in images])/255
stop = time()

print('Shape', pixels.shape)
print('Elapsed time', stop - start)

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
    plt.subplot(2,5,cnt+1)
    plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title(labels[idx]), plt.axis(False)

### Step 2
Run the training with one single neuron (as we did in the lecture). What accuracy can you achieve?

In [ ]:
inputs = Input(shape=(pixels.shape[1],))
outputs = Dense(1, activation="linear")(inputs)
model = Model(inputs, outputs)

model.summary()

# Remember the (random) initial weights, we will need them later
weights_init = model.layers[1].get_weights()[0].copy()    # copy, since the training changes the weights
keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(pixels, labels, epochs=25, batch_size=32, verbose=2)

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss');

In [ ]:
# Run it on the entire dataset
predictions = model.predict(pixels, verbose=0).squeeze()
predictions = (predictions > 0.5).astype(int)
correct = 0
for prediction, label in zip(predictions, labels):
    if prediction == label:
        correct = correct + 1

print('Accuracy', correct/len(labels))

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)

In [ ]:
wrong = np.where(predictions != labels)[0]
print('Number of mistakes', len(wrong))

for cnt, idx in enumerate(wrong[:10]):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)

### Step 3
Make further modifications to improve the accuracy (e.g. add more neurons, more layers, etc.). What is the maximum accuracy you can achieve?

*Note: the accuracy you compute here is measured on the same images the model was trained on. We will discuss how to evaluate a model properly in one of the next lessons.*


### Notes and answers
I came to the course from ML/DL background. So, I didn't know about pixel operations, but NNs are a comfortable topic for me :)

Since we're playing an overfitting game, I've just added a simple but complete MLP for binary classification (with binary cross-entropy loss and sigmoid in the outputs) with 3 hidden layers. Because we have around 3x more params I've spent more time training the model (100 epochs vs 25 in the 1-neuron example). With this slightly more sophisticated architecture, I've achieved better accuracy (0.98 vs 0.935).

In [ ]:
inputs  = Input(shape=(pixels.shape[1],))
hidden1 = Dense(3, activation="relu")(inputs)
hidden2 = Dense(5, activation="relu")(hidden1)
hidden3 = Dense(3, activation="relu")(hidden2)
outputs = Dense(1, activation="sigmoid")(hidden3)
model = Model(inputs, outputs)

model.summary()

# Remember the (random) initial weights, we will need them later
keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy')
history = model.fit(pixels, labels, epochs=100, batch_size=32, verbose=2)

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss');

In [ ]:
# Run it on the entire dataset
predictions = model.predict(pixels, verbose=0).squeeze()
predictions = (predictions > 0.5).astype(int)
correct = 0
for prediction, label in zip(predictions, labels):
    if prediction == label:
        correct = correct + 1

print('Accuracy', correct/len(labels))

In [ ]:
for cnt, idx in enumerate(np.random.randint(0, len(images), 10)):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)

In [ ]:
wrong = np.where(predictions != labels)[0]
print('Number of mistakes', len(wrong))

for cnt, idx in enumerate(wrong[:10]):
    plt.subplot(2,5,cnt+1), plt.imshow(images[idx], cmap='gray', vmin=0, vmax=255)
    plt.title('Label: ' + str(labels[idx]) + ' | Prediction: ' + str(predictions[idx]))
    plt.axis(False)

### Bonus question
Stack two ```Dense``` layers with ```activation="linear"``` on top of each other. Do you get a better model than with one single neuron? Why (not)? Hint: have a look at the slide about the neural network and non-linearity.

In [ ]:
inputs  = Input(shape=(pixels.shape[1],))
hidden1 = Dense(3, activation="linear")(inputs)
hidden2 = Dense(5, activation="linear")(hidden1)
hidden3 = Dense(3, activation="linear")(hidden2)
outputs = Dense(1, activation="linear")(hidden3)
model = Model(inputs, outputs)

model.summary()

# Remember the (random) initial weights, we will need them later
keras.utils.plot_model(model, to_file="model.png", show_shapes=True, show_layer_names=True)

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')
history = model.fit(pixels, labels, epochs=100, batch_size=32, verbose=2)

In [ ]:
# Plot training history
h = history.history
epochs = range(len(h['loss']))
plt.plot(epochs, h['loss'], '.-'), plt.grid(True)
plt.xlabel('epoch'), plt.ylabel('loss');

In [ ]:
# Run it on the entire dataset
predictions = model.predict(pixels, verbose=0).squeeze()
predictions = (predictions > 0.5).astype(int)
correct = 0
for prediction, label in zip(predictions, labels):
    if prediction == label:
        correct = correct + 1

print('Accuracy', correct/len(labels))

### Notes and answers
Since we throw in more weights, we get better accuracy because the model has more params to build a better fit. It works here; the result is better than for one neuron, but worse for an MLP with non-linearity. If we did the math, we could build an equivalent network with just one layer having more neurons, and that's what happens in the background. Because the model is linear, all hidden layers collapse into one layer.